$$\text{Pruebas Metodología ABSA}$$
$$\text{Análisis de sentimientos}$$
$$\text{Encuesta de Calidad}$$

> ***Objetivo***: Probar desempeño de combinaciones de modelos para las siguientes tareas:

- División de respuesta en fragmentos de texto - una idea por fragmento (modelos spacy)
- Categorización de fragmentos de texto en aspectos predefinidos (modelos Zero Shot)
- Asignación de sentimiento según aspecto (Modelos BERT para análisis de sentimientos)

> ***estructura iteraciones***


| Estado | N° Iteración | lista aspectos | Conectores | Modelo Spacy | Modelo Aspectos | Modelo Sentimientos |
| ------------ | ------------ | ------------ | ------------ | ------------ | ------------ | ------------ |
| Ejec | # 1 | ASPECTOS_CANDIDATOS_CALIDAD_v1 (OAC) | CONECTORES_v1 (manuales Cris) | `es_core_news_lg` | `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` | `finiteautomata/beto-sentiment-analysis` |
| Prog | #2 | ASPECTOS_CANDIDATOS_CALIDAD_v2 (eq PRISMA) | CONECTORES_v1 (manuales Cris) | `es_core_news_lg` | `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` | `pysentimiento/robertuito-sentiment-analysis` |


# Configuraciones Globales

In [ ]:
%pip install spacy transformers torch

In [ ]:
# Imports
import re
import spacy
from transformers import pipeline

In [ ]:
# Para correr en GPU

import torch

# Selecciona GPU si está disponible, si no cae a CPU
device = 0 if torch.cuda.is_available() else -1
print(f"Dispositivo: {'GPU (cuda:0)' if device == 0 else 'CPU'}")

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
# Instalación modelo spacy
import sys
!{sys.executable} -m spacy download es_core_news_lg

In [ ]:
# Lista 'comentarios_extensos' con los 20 comentarios más largos para hacer las pruebas

# comentarios_extensos = [
#     "ABRAN LAS TERRAZAS. ABRAN LAS TERRAZAS. ABRAN LAS TERRAZAS.",
#     "Considero que la universidad y el programa académico tienen una muy buena calidad en términos generales. A lo largo de mi carrera he aprendido muchísimo y le guardo un gran cariño a la institución, especialmente por el nivel académico y las oportunidades de formación que ofrece. Sin embargo, también considero importante expresar algunas inconformidades de manera respetuosa y constructiva. En primer lugar, siento que el costo de la matrícula ha aumentado de manera desproporcionada cada año. Actualmente, pagar cerca de 19 millones de pesos representa un esfuerzo enorme para muchas familias, y personalmente no percibo que ese incremento se vea reflejado en mejoras significativas dentro de la universidad. Por ejemplo, Exterdata, que parece ser una de las áreas con mayor inversión tecnológica, constantemente presenta fallas que interrumpen la continuidad de las clases y dificultan el aprendizaje. Asimismo, las rutas universitarias muchas veces no dan abasto, y algunas sillas de los edificios G y F se encuentran demasiado deterioradas, incómodas e incluso en condiciones poco agradables visualmente. Desde el punto de vista académico, considero que la universidad tiene un muy buen nivel; sin embargo, creo que deberían revisarse algunos criterios de evaluación de ciertos docentes. En ocasiones se realizan parciales escritos con preguntas extremadamente extensas y complejas para tiempos muy limitados, lo cual termina evaluando más la velocidad que realmente el conocimiento. Además, algunos profesores, debido a sus otras obligaciones laborales, tienen poco tiempo para los estudiantes: faltan a clases, no cuentan con espacios suficientes de monitorías o incluso muestran molestia cuando los estudiantes solicitan apoyo académico. Por otro lado, considero que existe una baja inversión en el desarrollo académico extracurricular dentro de algunas áreas, especialmente en Finanzas. Existen muchas oportunidades valiosas como visitas a bancos, entidades financieras, empresas o instituciones como el Banco de la República, pero durante mi carrera nunca tuve acceso a este tipo de experiencias organizadas por la universidad. Esto contrasta con otros programas que sí cuentan con proyectos de alta inversión y experiencias financiadas institucionalmente, como ocurre en algunos espacios de Gobierno y Relaciones Internacionales. Sería muy valioso que también se fortalecieran este tipo de oportunidades para estudiantes de Finanzas. Finalmente, también considero que debería existir un mayor acompañamiento en temas relacionados con intercambios estudiantiles y becas. En mi experiencia, el manejo de tiempos y la orientación brindada no fueron los mejores, y esto terminó limitando posibilidades que tanto mi familia como yo esperábamos poder aprovechar durante mi formación universitaria. En términos generales, valoro muchísimo todo lo que la universidad me ha enseñado y reconozco el impacto positivo que ha tenido en mi formación personal y profesional. Precisamente por ese aprecio, considero importante señalar estas deficiencias, ya que son aspectos que podrían mejorarse para fortalecer aún más la experiencia estudiantil y motivar a más personas a ingresar al programa en el futuro.",
#     "Estructuralmente la universidad cuenta con las condiciones para el desarrollo de las actividades académicas. Los espacios son pensados para tener las garantías de comodidad y conexión necesarias, sin embargo la infraestructura para la habitabilidad es excluyente y normada, los espacios de encuentro para grupos grandes son escasos y se limitan a escaleras o corredores poco amenos. La convivencia está pensada desde la individualidad, no permite el encuentro y la reflexión de la universidad desde la misma universidad porque la impide. El acceso a esta misma tampoco está pensada desde la inclusión. Los espacios para la movilidad de personas con dificultades físicas es nula, así como tampoco las garantías para que el buen desarrollo académico fuera de la institución se dé pese a las condiciones de restricción física que puedan tener algunos estudiantes. No hay suficientes recursos humanos para atender las situaciones psicosociales, de modo que el sistema de atención psicológica que tiene la universidad funciona igual que una EPS, es escaso y no logra tener un impacto tanto en la demanda como en el mejoramiento de la calidad de vida de las demandas de quienes asisten a sus espacios. La proyección de la universidad en términos de su visión está relegada a los intereses administrativos, una universidad pensada para la proyección social y la construcción de una sociedad más democrática que se dirige por los intereses enteramente económicos termina, como efectivamente lo hace, ignorando las necesidades y aspiraciones de un estudiantado que al día de hoy es heterogéneo, que viene de contextos socioculturales distintos a ser hijos de académicos, políticos o empresarios acomodados sobre el estrato 4, sino que son hijos de obreros y obreras de las periferias que a sabiendas de ser financiados con recursos tanto distritales como privados no esperan ser asimilados en una educación elitista que se intenta reproducir desde las formas administrativas de la universidad. Los espacios de clase con la planta docente que cuenta antropología al día de hoy permiten tener las discusiones integrales, conscientes de la subjetividad y del lugar de poder que se habita en la universidad y que vamos a habitar nosotros en la sociedad, tener docentes críticos, reflexivos, que enseñan desde las múltiples posibilidades de mundo, y que reconocen las estructuras de poder que constituyen el mundo, y a la universidad como parte de ese mundo. Vale la pena pensarse en las oportunidades reales de que la universidad sea realmente más liberal, si le apuntan al desarrollo propio de quienes habitan este espacio, de la posibilidad de que, atendiendo a las nuevas dinámicas que ha venido aceptando la universidad por la coyuntura de la educación superior en Colombia, que le ha hecho abrir las puertas a poblaciones populares gracias a convenios firmados y financiados, y que asuma no una actitud paternalista de dar las cosas, pero sí una actitud liberal de abrir las posibilidades en las que puede ser construida la universidad desde la posibilidad de habitarla de otras maneras distintas a la del aula y las calificaciones.",
#     "Deberían realizar una pedagogía completa sobre la modalidad de profundización del programa. Muchos profesores desconocen la modalidad y así mismo, no se enfocan en el desarrollo de la profundización, sino que se enfocan en la investigación. Si se abre una modalidad de profundización es para mejorar la oportunidad de adquirir conocimientos más avanzados y especializados en un campo determinado no para centrarse en la investigación. (Ya existe una modalidad que se centra en eso). Así mismo, es incoherente que se establezcan los mismos criterios de modalidad de grado como en la modalidad de investigación. Ejemplo: hacer tesis, artículo científico, capítulo de libro en una modalidad de profundización, es lo mismo que hacerla en la de investigación, por lo que no tiene sentido escoger una modalidad u otra. A razón de lo anterior se sugiere que se creen nuevas modalidades de grado que se enfoquen en el uso práctico de conocimiento de la profundización que ofrece el programa y que no sean las mismas que ofrece la modalidad de investigación. Si la razón de que no se puede evitar el tema investigativo es por el convenio con Francia y que tradicionalmente está ligado a la investigación, entonces que exista una cláusula que exime al programa en la modalidad de profundización con los beneficios de la doble titulación con Francia y demás. Que quede a criterio del estudiante si desea aplicar a la doble titulación con Francia para que no quede atado a lo que implica el convenio. En síntesis, se busca que se fortalezca la modalidad de profundización que ayuda a reducir la alta tasa de deserción del programa, más aún que la mayoría de maestrantes son de la carrera diplomática y que por razones obvias del servicio, no se pueden dedicar a una investigación profunda (ya que toda investigación requiere tiempo y recursos). La Universidad no puede seguir desconociendo la realidad de los maestrantes de la Academia diplomática. Además, se debe mejorar mucho en el tema administrativo y acompañamiento a los estudiantes. En primer lugar, casi se acaba este tercer semestre y no tenemos acceso al correo electrónico, ni tampoco tenemos acceso a la biblioteca y recursos bibliográficos. (Estas herramientas son vitales para llevar a cabo la modalidad de grado). En segundo lugar, no se comprende por qué no se asignan tutores a los estudiantes. Ese proceso de búsqueda también es incoherente, ya que los mismos profesores se pueden negar a ser tutores, teniendo en cuenta que los temas no pueden ser de su interés para el desarrollo de sus propias investigaciones. Es decir hay intereses académicos de por medio. La Universidad es una institución que guía y brinda las herramientas para el aprendizaje, con el criterio anterior debe de brindar al profesor adecuado para la modalidad de grado escogida. El estudiante no debería estar ofertando su modalidad de grado para que algún profesor se interese en colaborarle. Espero pueda seguir mejorando el programa, en especial la modalidad de profundización ya que muchos están interesados en esta modalidad.",
#     "Amo la universidad, es mi segundo hogar, pero sin duda la embarramos como facultad escogiendo decano, es horrible. En la mayoría de lo que llevo de carrera me he sentido sola en el proceso, estoy haciendo doble programa con FIGRI y sin duda allá los acompañan más, incluso me responden mejor ellos las dudas sobre el doble programa que mi facultad de base (economía), hay palanca para casi toda oportunidad no sé si así debe ser, pero al menos la institucionalidad debería reducirla, no promoverla; hay personas que son promotoras y monitoras al tiempo quitándole la oportunidad a los demás de tener opciones para agregar en su hoja de vida, los criterios de selección de las monitorias no son claros y ahora toca pasar por el filtro del coordinador sin saber cuáles son sus razones, generalmente escogen a las mismas personas una y otra vez. La relación de la facultad con otras universidades es nula, solo nos integramos con otras universidades en eventos grandes y casi nunca hay; en temas de investigación con otras universidades solo está el cane y como tal no es investigación sino competencia, en intercambios lo único que promueven es Francia. El decano no escucha y ya no hay una línea clara de alguien que nos escuche, podemos tener un mal profesor y debemos seguir con él porque para que lo cambien es demorado, de hecho, para cualquier queja el trámite es largo y no llega a nada. Son incompetentes en varios aspectos desde llevando el registro de horas bienestar hasta darte una cita con el coordinador que se la pasa más ocupado en eventos que poniéndonos atención, en mi facultad la mayoría está perdido y quien no lo está lo echan, parece que nadie escucha, las charlas que ponen solo hay hombres exponiendo, la cantidad de profesoras que hay es mínima (creo que de planta solo hay tres), los representantes son malos y no hay más sistemas que permitan decir las inconformidades sin que uno quede perjudicado, no hay unión con la práctica, no hay salidas de campo o algo parecido, los profesores faltan y nadie les dice nada, algunos ni siquiera dan la cátedra completa, hay solo dos jóvenes investigadores (aunque lo positivo es que sacaron a los que llevaban años ahí), las becas por excelencia las recortaron y no le avisaron a nadie. Lo irónico es que nos dictan economía institucional y mi facultad es un ejemplo perfecto de cómo una mala institucionalidad afecta. Lo bueno es que la mayoría de los profesores son muy buenos (excepto cuando la facultad saca a los buenos para poner a profesores por palanca), la educación es diversa, es decir, conocemos todas las posiciones o al menos la mayoría sobre un tema, te dejan escoger qué posición quieres tomar. La tecnología y los espacios son preciosos; no puedo parar de querer que toda persona que conozco vaya a mi universidad. A pesar de todo recomendaría al Externado, tal vez no meterse a economía como tal (por ahora), pero sí a las demás carreras.",
#     "1. Muchas de las preguntas en este cuestionario no las conocía. Durante el programa de la especialización, nunca se nos informó sobre las instancias del estudiante o del profesor, ni sobre programas de intercambio o la posibilidad de participar en programas de educación con otras universidades. Tampoco se mencionaron opciones de investigación, ni conocimos laboratorios o aulas de apoyo para lo que veíamos en clase. 2. En la pregunta relacionada con la calidad del programa, califiqué como insatisfecho porque el plan de estudios no estaba actualizado a las tendencias actuales del mundo digital. Además, las materias no estaban relacionadas entre sí, lo que generaba repetición de contenidos. En varias asignaturas (Marketing digital 1, experiencia de usuario, transformación digital, neuromarketing) vimos los mismos temas: funnel, journey map, buyer persona, sin secuencialidad. Claro, son temas muy amplios que seguramente en unas materias hubiéramos visto etapas del funnel o del journey pero no fue así, siempre vimos los mismos temas. Daba la percepción de que la universidad no revisó o entregó la estructura de las clases a los profesores. 3. Las materias optativas ofrecidas no estaban alineadas con lo que esperábamos. Tanto así que tuvimos que proponer nuevas asignaturas para cursar. 4. En el primer módulo hubo materias que no se relacionaban directamente con el propósito del programa, como desarrollo sostenible. Aunque los temas fueron interesantes y nos concientizaron sobre la situación del planeta, no encontré una conexión clara con el mundo laboral en áreas digitales, marketing, o los sectores en los que la mayoría de los estudiantes nos desempeñamos. Igualmente, en la materia de liderazgo, es un tema tan relevante para los roles que encontramos hoy en el mercado, pero lo que vimos en esta materia no agregó valor. Sentí que perdimos tiempo con ejercicios como jugar Sudoku o con legos. Tenía expectativas de aprender a enfrentar entornos laborales complejos, generar ambientes productivos y dar buen feedback, pero lo que vimos fueron actividades más apropiadas para compartir en ambientes familiares o amigos. Los tiempos asignados a estas materias se pudieron haber utilizado con temas más afines al ser de la especialización. 5. Destaco: 1. Materias muy buenas y profesores excelentes, como por ejemplo: analítica digital, marketing digital II, pensamiento estratégico, entorno y competitividad, neuromarketing. También fue muy interesante los seminarios internacionales (fueron muy cortos). 2. Las instalaciones de la universidad sin dudas son muy lindas, los espacios de estudio, las rutas, la cafetería (podrían ajustarse los horarios de break para evitar largas filas y llegar tarde a las clases). Para cerrar, tenía expectativas más altas del programa, dada la reputación de la universidad. Gracias.",
#     "Me gradué en 2020 y puedo afirmar que el programa ofrece una excelente calidad académica, así como un cuerpo docente altamente capacitado. La formación que recibí fue integral y pertinente, abordando temas relevantes para el contexto educativo actual en Colombia en todos los niveles de formación. Los conocimientos adquiridos han sido fundamentales para mi desarrollo profesional y personal. Sin embargo, considero que el programa presenta algunas limitaciones en cuanto a su flexibilidad. Las opciones de electivas son escasas y, en su mayoría, son casi obligatorias. Esto reduce la posibilidad de diversificación y personalización del aprendizaje, que es crucial para adaptarse a las diversas necesidades y contextos de los estudiantes. Además, sería beneficioso establecer una integración de doble titulación con otros programas de la facultad, especialmente con la Maestría en Evaluación y Aseguramiento de la Calidad de la Educación, ya que las temáticas de ambos programas están estrechamente relacionadas y podrían complementarse de manera efectiva. Otro aspecto importante es la necesidad de mantener una comunicación más fluida con los egresados. Sería valioso recibir propuestas para seguir vinculados a las líneas de investigación del programa, así como conocer ofertas laborales y académicas que faciliten nuestra ruta formativa hacia el doctorado. La conexión con los egresados no solo fortalecería el sentido de comunidad, sino que también permitiría un intercambio enriquecedor de experiencias y conocimientos. En la actualidad, parece que solo se nos considera para procesos de autoevaluación con fines de registro calificado y acreditación. Sin embargo, los egresados tenemos mucho que aportar desde nuestra labor docente y nuestra experiencia en la práctica educativa. Un mayor apoyo por parte de la facultad podría enriquecer significativamente el currículo y fomentar la innovación en la enseñanza. Por último, considero que la internacionalización del currículo es un aspecto que debería ser prioritario. La falta de un enfoque en este sentido limita las oportunidades de los estudiantes para ampliar sus horizontes académicos y profesionales. Implementar estrategias que fomenten la internacionalización en el plan de estudios no solo beneficiaría a los estudiantes, sino que también posicionaría al programa de manera más competitiva a nivel global. Asimismo, el contacto con el sector empleador es prácticamente inexistente. Establecer relaciones más sólidas con empleadores permitiría no solo obtener retroalimentación sobre el programa, sino también enriquecer la formación con perspectivas del mercado laboral, lo cual es esencial en un mundo en constante cambio. Cordialmente Cristian Andrés Rojas Jiménez.",
#     "Yo siempre me he quedado que muchos profesores no saben enseñar, un docente no solo se contrata por lo que haya estudiado y lo que sabe, sino por cómo sabe enseñar, no sé si lo hacen pero es bueno que se les haga retroalimentaciones a los docentes de lo que pueden mejorar, que no sean soberbios, todos llegamos a esta vida a aprender y a aceptar los comentarios constructivos, así que hacer seguimiento de los docentes, de su forma de enseñar, que no sea a punta de teoría porque así poco se aprende, se debe ser más práctico en absolutamente todas las materias, ir, conocer, mirar, tocar, hacer a la par del docente, comparar si lo que estoy haciendo me tiene que dar así, la calidad educativa de la Universidad depende mucho de lo que enseñe el docente, de contar y aprender de su experiencia. Y de realmente aprender algo, muchas materias se enfocan en la nota pero no en realmente aprender, entonces no sirve para nada, en la evaluación docente se debería preguntar si el estudiante realmente aprendió algo o no y ya se convierte en otro punto a evaluar y es más fácil hacer seguimiento y mejora continua. En muchas materias durante estos 5 semestres me ha pasado que el profesor no hace interesante la materia, los temas no tienen objetivo por el cual estar aprendiendo eso y se pierde la motivación, ahora, si se hiciera un evaluó de la motivación de los estudiantes por materia también sería un punto para la medición del docente; muchas veces he expresado que mi matrícula ha sido perdida de tiempo y de dinero porque los docentes no saben enseñar, no se centran en dejar un verdadero conocimiento sino en sacar nota porque sí, porque toca cumplir, porque no hayan otros métodos de evaluación. Este semestre he tenido la oportunidad de tener parciales prácticos y es ahí cuando me doy cuenta de todo lo que realmente aprendí, existe un proyecto integrador pero también se ha convertido en una nota y ya, no en el sentido por el que realmente se creó, los docentes deben saber evaluar qué piden en esos proyectos para asegurar que sí se esté haciendo aterrizado a la vida real y que seamos capaces de hacerlo en la vida real y que sea algo relevante. Concluyendo, considero que falta medición y seguimiento a los docentes, comprendo que es de libre cátedra, pero si se hace necesario que se revise y se tomen seriamente en cuenta todo lo que 'enseña' el docente y cómo lo hace.",
#     "Considero que hay muchas cosas mal con el problema: - Sociología está fragmentada hace mucho tiempo porque los profes toman partido con los estudiantes con tensiones personales y eso se nota en las clases. Además son imparciales presuntamente, pero cuando los estudiantes les quieres pedir ayuda si son capaces de tomar partido. - Hay profesores que ya deben salir y cumplen con su ciclo de pensión, porque es increíble que se deje a maestros que no actualizan sus metodologías y que tampoco se abren a nuevas propuestas. Además es un docente que tiene 5 clases y las da todas iguales, nadie aprende nada, es agotador y está acaparando empleos que pueden ser para maestros que están esperando la oportunidad de entrar con nuevas ideas. - Hay profesores que no cumplen con los requisitos para estar en la universidad y ya se ha pedido revisión constable. Este programa son amigos de amigos y han sacado a profes como a Fernanda Fierro por dejar a personas que no cumplen con los requisitos y tienen muchas más quejas. - Con el cambio de pensum a quienes quedamos con el programa anterior no se nos ha asegurado la calidad de la titulación, pues nuestro programa ya no aparece con alta acreditación solo el de 9 semestres. - Hay muy poca financiación para el programa y la articulación de los eventos es muy poca. - Las becas por excelencia académica para sociología son muy pocas y se entregan de forma arbitraria. Cómo es posible que personas que viajan la mitad del semestre lleguen y sean calificadas de la misma forma, cuando a quienes trabajamos los profesores nos ponen problema por no llegar un día y nos bajan la nota de participación arbitrariamente. Asimismo, una persona que solo está viendo la mitad de las materias o incluso solo dos no debería poder ganarse la beca, ya que la carga académica no es la misma. - Especialmente con las clases del profesor Fresnesda ese problema de las calificaciones es constable, al igual que la falta de actualización de los contenidos en clase y los comentarios que hace a las personas. Porque para él solo cierto tipo de condiciones diferentes son válidas, y por más que uno haga buenas aportaciones califica la participación solo por hablar. Si uno falta es un 2 automático así ese día nadie diga nada interesante y aunque se le ha pedido revisar su clase, su metodología y sus formas de calificación. Todo desde primero está igual.",
#     "Estudiantes: En cuanto a las medidas disciplinarias y sanciones del plagio considero que la facultad se ha vuelto alcahueta y desde el consejo directivo no se está reforzando el hecho de que los estudiantes cumplan a cabalidad con esto. Antes era blanco o negro, ahora con las pruebas en contra del estudiante, es gris y se mantiene al estudiante en el programa por miedo a que vengan demandas o sanciones de entes externos. En cuanto a los requisitos de permanencia, promoción y grado es el soberano colmo que el consejo directivo se pase por encima el reglamento. No se explica cómo a un estudiante se le dan dos reintegros o a otro se le permita ver por 5ta vez una materia. El nivel de exigencia debe venir desde el consejo, sin esto, los profesores NO tenemos herramientas. Profesores: En cuanto a las políticas de estímulos, reconocimientos y distinciones no existen o no se ven, o se ven cada 5 años cuando se celebran los lustros de la Universidad y la verdad no veo el mérito de llevar 20 años dando clase con excelentes resultados y tener la misma escala salarial de alguien que lleva 3 meses y no se conocen sus resultados. Es una mentira esto de los estímulos, reconocimientos y sanciones. Aunque debe decir que este año sí recibí un estímulo para firmar un otro sí a mi contrato. El estímulo, dicho por una persona del equipo rectoral fue 'si no lo firma, será mal visto'. Igual esto no lo lee nadie y es un saludo a la bandera, pero acá les queda. En cuanto a las políticas y programas de desarrollo profesoral se tienen que si se es doctora o candidato a doctor, se les retiene y se les da, sí o sí un sueldazo con contrato a término indefinido, no importa si son pésimos profesores, si no dictan en el pregrado o, mejor aún, si son solo investigadores que no producen ni vergüenza. En cuanto a los criterios para la evaluación del desempeño profesoral quiero decir que son clarísimos, solo que cuando no se cumplen y se quiere echar al profesor, algo aparece y esa persona puede seguir porque la protege cualquier fuero, lo que nos desmotiva a los profesores que somos buenos y que nos esforzamos por hacer las cosas verdaderamente bien. Lo que me lleva a decir que los criterios y mecanismos para la evaluación de profesores(as) no son ni eficaces, ni equitativos ni transparentes.",
#     "1) LA UNIVERSIDAD NECESITA UNA MEJORA FORMAL, MATERIAL Y CONTINUA EN LOS ESPACIOS DE RECREACIÓN, DEPORTE Y CULTURA, PUES AUNQUE ESTE EL ALCAZAR ES DEMASIADO LEJOS PARA PODER DISFRUTAR DENTRO DEL CAMPUS EN TIEMPOS LIBRES; 2) HAY QUE MEJORAR LOS DERECHOS DE LOS ESTUDIANTES, REBAJAR LOS PRECIOS EXCESIVOS DE LOS PREPARATORIOS Y REDUCIR LOS REQUISITOS (ABSURDOS POR LA CANTIDAD DE ESTOS) DE GRADO PARA LOS ESTUDIANTES DE DERECHO, (PUES YA SE HAN CURSADO 5 AÑOS DE MATERIAS Y PAGADO MÁS DE $100.000.000 MILLONES DE COP, COMO PARA SEGUIR PAGANDO MÁS Y NO PODER EJERCER O TRABAJAR, SOLO POR NO CUMPLIR CON UNOS REQUISITOS DE RELLENO). 3) EL TRATO CON LOS PROFESOR-ESTUDIANTE ES MUY DISTANTE Y A LAS DIRECTIVAS NO LES INTERESA EL BIENESTAR DE LOS ESTUDIANTES, DADO QUE SOMOS UN NÚMERO MÁS QUE SOLO INTERESA SI SE ESTÁ PAGANDO UNA MATRÍCULA, PORQUE AL FIN Y AL CABO LAS UNIVERSIDADES PRIVADAS SON UN NEGOCIO MUY REDONDO. ADEMÁS ME GUSTARÍA RESALTAR QUE EL MÉTODO DE ENSEÑANZA ES EL MISMO QUE HACE 200 AÑOS (NO HAY USO DE HERRAMIENTAS PRÁCTICAS, TECNOLÓGICAS O INNOVADORAS AL MENOS EN DERECHO, SOLO ES UNA PERSONA HABLANDO, CUANDO FACILMENTE ESO LO PUEDO ENCONTRAR EN LOS LIBROS, JURISPRUDENCIA Y DEMÁS). 4) LA UNIVERSIDAD EN DERECHO NO PERMITE HACER DOBLE TITULACIÓN EN DERECHO CON LA VAGA Y BÁSICA RESPUESTA QUE DERECHO ES MUY DIFÍCIL, POR LO TANTO SALIMOS MENOS COMPETENTES Y FUNCIONALES AL MUNDO LABORAL. 5) NO HAY UN PROGRAMA DE CONEXIONES LABORALES PARA LOS RECIÉN EGRESADOS; 6) LA UNIVERSIDAD ÚNICAMENTE NO DEBERÍA ENSEÑAR UN PENSUM EN LA CARRERA CORRESPONDIENTE, SE PUEDE ENSEÑAR COSAS MÁS ÚTILES Y PRÁCTICAS QUE PUEDAN SERVIR PARA LA VIDA PROFESIONAL O PERSONAL, TANTO QUE DICEN LA UNIVERSIDAD ENSEÑA DE FORMA INTEGRAL, PERO REALMENTE ESO NO SE VE, SOLO ENSEÑAN MATERIAS DE DERECHO, POR EJEMPLO NO ENSEÑAN CÓMO UNO DEBE VENDERSE COMO ABOGADO, CÓMO RELACIONARSE BIEN O POR EJEMPLO CÓMO SE PUEDEN COMPLEMENTAR Y RELACIONAR MIS HABILIDADES DE ABOGADO CON OTRAS ÁREAS O DISCIPLINAS.",
#     "El equipo docente del programa de Antropología es ejemplar, sin embargo, el alcance de sus labores se ve limitado por la falta de apoyo por parte de la Universidad. Por otra parte, la flexibilidad debe tener en cuenta el apoyo económico o facilidades para las y los estudiantes con el objetivo de que puedan finalizar sus carreras. La deserción estudiantil se materializa por el abandono institucional, resultan ser las y los docentes quienes acompañan emocionalmente a algunos estudiantes en medio de sus posibilidades, pero bienestar universitario es inexistente y el acompañamiento que realizan no es suficiente para evitar la deserción. Por consecuencia la posibilidad de hacer doble titulación o un intercambio sigue siendo un beneficio de unos pocos. Las clases de prácticas y pasantías no son adecuadas en términos de la carga académica para los últimos semestres. De igual forma, no hay un portafolio de posibilidades a partir de las alianzas que realice directamente la Universidad, queda a potestad del estudiante lo que pueda conseguir. Una vez finalizados los estudios universitarios y antes de esto, no existe un acompañamiento real para que las personas puedan insertarse al mercado laboral, lo que deja a muchas y muchos sin experiencia laboral frustrados y aceptando ofertas laborales que no cumplen con el perfil y que son salarios muy bajos. Por todo lo anterior, es fundamental generar más opciones de participación, reconocimiento, apertura a convenios y otros escenarios que permitan que las y los estudiantes puedan generar los insumos para materializar lo que están aprendiendo, pero también, con miras a vincularse laboralmente y/o continuar con la producción académica. Asimismo, es fundamental reconocer las posibilidades y condiciones de vida de las y los estudiantes, porque muchas personas dejan sus estudios académicos o se demoran años en graduarse por abandono de la Universidad.",
#     "Uno espera que la maestría se base en casos prácticos, la mayoría se hizo en aplicación de lo teórico y retórica que en libros se puede encontrar, mucho del contenido visto se discutieron temas ya conocidos sin ir a lo específico, el contenido fueron brochazos de un temario sin ir a la profundidad que es lo que el estudiante espera, además la falta de comunicación con los temas de diferentes profesores; ejemplo de ello es, en materias diferentes se tratara los mismos temas y eso conllevó que lo recibido no fue lo esperado, resalto materias frente a otras tales como: 1. En uso de teoría aplicado a la práctica 2. Optimización estocástica 3. Profundización en Analytics data 4. Profundización en Python financiero (1 profesor) y Análisis de riesgo e incertidumbre (2 profesores) 5. Opciones reales y Fortalecimiento de Finanzas (misma profesora), esta última se pidió y quedó una profundización pendiente que a la fecha no se ha dado (27/02/2024) ya que las clases recibidas en primer módulo (2 materias) fueron muy estilo pregrado y el segundo módulo (1 materia) el contenido dictado no era el apropiado para ser visto en Maestría. 2. Valoración humana aplicado a negocios 3. Desarrollo sostenible 4. Entorno y competitividad 5. Análisis organizacional (profesor Jimmy) 3. Mezcla de herramientas y componente humano 4. Innovación en proyectos. Estas materias y sus expositores fueron a mi expectativa, lo mejor de todo el programa y no por ello debo agradecer tanto a la universidad por seleccionar un personal que ama lo que quiere expresare a los estudiantes y a ellos mismos por su gran contenido y humanidad. Todo lo anterior quiero dejar constancia que para ser una maestría de 26 materias y solo 9 de ellas cumplieron con altas expectativas, es decir el 34.61% del 100% me queda una sensación de frustración. Gracias y espero tomen a consideración este análisis.",
#     "Considero que la logística del plan de estudios de la especialización podría mejorarse, especialmente en la distribución de las clases. En varias asignaturas, como Persuasión, Opinión Pública y Terrorismo, los profesores no lograron completar su cátedra, mientras que en otras, como Investigación Formativa, hubo sesiones en las que no se abordó información relevante. Además, el tiempo destinado para diseñar una estrategia psicopolítica es bastante corto, lo que dificulta su desarrollo. Algunos estudiantes que trabajan en instituciones gubernamentales u organizaciones con enfoque en estos temas tuvieron mayores facilidades para implementarla. Sin embargo, otros, como en mi caso, no contamos con ese respaldo, lo que hizo que el proceso fuera más complejo. Por ello, sugiero que la Universidad establezca convenios que permitan implementar estos proyectos en escenarios reales, lo que contribuiría a lograr un mayor impacto y efectividad. Asimismo, propondría incluir en el plan de estudios asignaturas relacionadas con procesos de paz, negociación y mediación, así como otros ejes temáticos clave para la práctica profesional. Si bien la teoría es fundamental, al tratarse de una especialización orientada a la construcción de estrategias dentro de comunidades, una mayor aproximación práctica resultaría esencial. Por último, considero que los estudiantes de posgrado no contamos con acceso suficiente a los espacios de bienestar ofrecidos por la Universidad. En mi caso, el único servicio que utilicé fue la ruta de transporte, y me hubiese gustado una mayor divulgación e inclusión en estos espacios. Sin embargo, debido a los horarios, no fue posible. Además, tuve poca información sobre becas, apoyos económicos, oportunidades para participar en semilleros de investigación o la posibilidad de realizar voluntariados o intercambios académicos.",
#     "Siento que muchas de las preguntas están enfocadas a pregrado. En mi experiencia como estudiante únicamente de posgrado, hay mucha información que no te dan y tampoco llegas a tener contacto o participación, por ejemplo, en el nombramiento de representantes estudiantiles de posgrados, salvo una elección que se realizó, recuerdo bien que son el representante de la especialización. Tampoco solíamos recibir mucha información de su parte. Tampoco hay mucha información que no se comparte o que se demoran en compartir. Por ejemplo, estuvimos varios meses sin poder acceder a bases de datos porque la capacitación que se debe hacer desde biblioteca (si no recuerdo mal eran ellos), demoró bastante, especialmente porque cuando nos la hicieron, muy pocos compañeros tenían un usuario habilitado y no le pudimos sacar provecho a eso. Tampoco tuvimos acceso oportuno a correo, las notas en algunos casos demoraron varios meses en cargarse, lo cual hizo que el proceso de grado fuera extenso, y por ejemplo tampoco recibimos una guía por el campus universitario, especialmente pensando en quienes no conocíamos las instalaciones previamente y quienes vienen de otras ciudades. Esto hace que ubicarse tome tiempo y a veces perdamos tiempo intentando llegar a clase. Entiendo que al asistir solo una vez al mes, mucha de esta información no se comparte con nosotros, pero estaría bueno que se diera. También estaría muy bien que cuando por ejemplo se ponen quejas contra algún personal, se informe a los estudiantes el resultado del trámite de dicha queja, para saber que están teniendo, cuando menos, en cuenta la opinión del estudiante. Como siempre, son recomendaciones que si se fijan, no tienen relación con la especialización como tal sino con un aspecto general de la universidad, son cosas que harían que la experiencia de hacer un posgrado en el Externado sea mucho mejor.",
#     "En mi opinión la facultad y el programa tienen 4 grandes debilidades. Y esto lo digo con el análisis de otras universidades, y con la intención siempre de que mi casa de estudios sea la que más resalte. 1. El Campus Virtual está muy desactualizado, no todas las clases lo tienen para siempre profundizar, hacer actividades interesantes e interactivas para los estudiantes. Necesitamos que el campus sea como nuestra U virtual, donde podemos acceder a todo pero en el sistema, teoría, actividades, que nos den puntos, notas, cursos adicionales, profundizar, etc. 2. Las notas nunca las tenemos a la mano en la APP de la Universidad, nunca se sabe nada hasta la última semana o terminado el semestre. 3. Los seminarios de profundización, materias entre otros, no solo debería ser guiado únicamente a Banca y Corporativo, debe ser más abierto. 4. La parte administrativa y directiva debería tener como prioridad mantener a los estudiantes activos con la universidad, que se sientan queridos y con el acompañamiento. Nosotros somos el motor de la universidad, pero nos tratan como simples seres que estamos ahí para pagar matrícula y sobrepasar los semestres, y de lo contrario simplemente se pueden ir. Eso no es así. Y los altos directivos como el Rector lo saben, pero el decano y otros mandos han desconocido un poco eso. El estudiante es indispensable, y hay que defender su integridad a toda costa, el mantenerlo en la U, el querer que mejore, la proposición de actividades interesantes, salir del ambiente académico de vez en cuando, incentivarlo a participar en las actividades de Bienestar. Entre otras muchas cosas (a mi me hubiera encantado ser parte del consejo directivo y tener más inherencia en todo eso para mejorar mi casa de estudios y mi facultad como se merece).",
#     "El programa le faltó mayor organización en todas las actividades. Mejor planeación para los encuentros presenciales, los cuales deben definirse desde el inicio y no un par de meses antes. De igual forma la coordinación del programa debe involucrarse más en las quejas de los estudiantes hacia el desempeño de algunos docentes, quienes regularmente quieren impartir metodologías no acordes con una maestría (realmente considero que tomar 10 calificaciones en 3 semanas es excesivo). El estudiante no se mide por la cantidad de puntos que obtenga en 10 actividades, si no por la forma como se apropie del conocimiento. Adicionalmente algunos profesores, no le dan la verdadera importancia que amerita el programa, presentándose sin preparar clases, algunos transmitiendo por zoom desde el celular, otros cargando las calificaciones 3 o 4 meses después de haber terminado el módulo. Estas son algunas de las malas prácticas que me llevan a decir que el programa no llenó las expectativas que en un inicio tenía. Agregando también que el descontrol final para el proceso de grado, en donde la facultad, el CIAT y la universidad se pasaban la pelota en cuanto a responsabilidades. La gestión del proceso de grado, un completo desastre, donde como estudiante me tocó llamar por más de 2 meses casi a diario, mandar muchos correos y pedir favores casi que rogando, para poder graduarme, lo anterior habiendo cumplido con todos los requisitos en tiempo. Y al final obtener un grado por secretaria. Esto no es una queja, es más bien una recomendación para que el programa mejore y a futuro puedan tener comentarios de felicitaciones, los cuales en esta ocasión me cuesta mucho hacerlos. Muchas gracias por su atención.",
#     "Las horas de Bienestar Universitario no deberían ser un requisito de graduación, teniendo en cuenta que muchas veces son imposibles de realizar por el horario de las clases del programa y si el supuesto propósito es darles a los estudiantes un espacio de 'bienestar' debería ser una elección propia, ya que, en vez de que los estudiantes le cojan gusto a los cursos le cogen fastidio. También muchas de ellas requieren un desplazamiento grande y en horas de la tarde noche (ejemplo natación etc.). Siento que hay carencia en cuanto a los cursos libres orientados al marketing o la comunicación, pues aunque reconozco que supuestamente es un espacio de dispersión de las materias de la carrera, la facultad debería ofrecer otros cursos libres orientados a los estudiantes que les gusta más el área de la comunicación y el marketing, pues los pocos que hay son en inglés o exclusivamente de investigación, historia o periodismo. En cuanto a la materia de Flujos Audiovisuales, siento que en la forma en la que está orientada (el desarrollo de una serie, ya sea ficticia - tema trabajado en audiovisuales 1 - o documental - trabajado en audiovisuales 2), debería ser una optativa, pues siento que los temas vistos son una reiteración complementaria a las materias del componente audiovisual anteriormente vistas (introducción audiovisual, audiovisual 1 y audiovisual 2). Y en todo caso más que ser una optativa, si esta ha de permanecer como materia obligatoria para los estudiantes de séptimo semestre, debería estar orientada a todas las formas de trabajo en la industria audiovisual no profundizadas hasta el momento como el área de la publicidad.",
#     "Ingresé al programa de especialización en Derecho Público con la expectativa de que el programa me mostrara el camino para poder cambiar de enfoque profesional y adentrarme hacia áreas del derecho que son de mucho mayor interés para mí que aquellas en las que, por cuestiones de la vida, me encuentro actualmente. Desafortunadamente, no encontré en el programa las oportunidades que estaba buscando, y más bien sentí que ingresé a un curso largo de profundización cuyo único propósito es transmitir los contenidos del programa, entregar el diploma y chao, te vi, sin ofrecer posibilidades a los estudiantes que quieran ahondar y formar parte de proyectos, semilleros u otro tipo de espacios en donde contemplar oportunidades de crecimiento. Entiendo que quizá era ingenuo de mi parte tener semejantes expectativas para una especialización, y que tal vez una maestría sea el medio más adecuado para satisfacerlas. No obstante, estoy seguro que de haber encontrado algo más de interés de parte de la Universidad de cultivar a sus estudiantes de especialización, yo sin duda ya habría hecho el esfuerzo de vincularme a una maestría. Por último, no es muy aconsejable programar clases en jornadas enteras, ya que esto, por un lado, dificulta que estudiantes que trabajamos podamos encontrar la flexibilidad necesaria en nuestros trabajos para asistir, y por el otro, genera un incentivo a que ciertos docentes de poca ética se aparezcan una hora o dos horas tarde a una clase de cuatro horas, y rellenen el poco tiempo restante con monólogos vacíos e improvisados para disimular la nula preparación de sus clases.",
#     "Agradezco a la Universidad Externado y especialmente al programa de Maestría en Pensamiento Estratégico y Prospectiva por el proceso de formación que me ha permitido asumir en los últimos años. Considero importante tres aspectos que el programa podría asumir y desarrollar: 1. Implementar seminarios teóricos y metodológicos del área disciplinar que relacionen los estudios de futuro. Si bien el programa es un posgrado en el nivel de profundización, considero importante acoger los nuevos desarrollos investigativos (Estado del arte) de los estudios de futuro que actualmente se desarrollen en diferentes regiones del orden global; además, las teorías y bases conceptuales de la prospectiva, son fundamentales para ubicar los orígenes, desarrollos y perspectivas de dicha disciplina para identificar las tendencias que pueden presentarse en los estudios de futuro. 2. Oferta de seminarios o cursos de actualización. Es comprensible la dinámica en los cambios que el mundo presenta de manera acelerada y como programa de posgrado, es importante convocar a sus egresados para tomar cursos cortos o seminarios que presenten temáticas o perspectivas de la disciplina que sean disruptivas y que tengan un soporte epistemológico para los estudios de futuro. 3. Homologación para el nivel doctoral. Reiterando que el posgrado no presenta un carácter en investigación, es importante que en un escenario de posibilidades la maestría acoja el cambio cualitativo en investigación para que pueda encadenarse con el programa doctoral. Agradezco su especial atención, Rolando Centeno Tapiéro MPEP26 3124605524.",
# ]

# Iteración 1

> ***Parámetros***:

ASPECTOS_CANDIDATOS:
- etiquetado manual OAC: `ASPECTOS_CANDIDATOS_CALIDAD_v1`

CONECTORES:
- Listado manual propuesto por Cristina

MODELOS:
- spacy: `es_core_news_lg`
- zeroshot: `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`
- Análisis sentimientos: `finiteautomata/beto-sentiment-analysis`

## Configuraciones iteración

In [ ]:
ASPECTOS_CANDIDATOS_CALIDAD_v1 = [
'Bienestar Universitario',
'Calidad académica',
'evaluación docente',
'métodos de evaluación',
'Monitorías',
'opciones de grado',
'Relación con egresados',
'Servicios Académicos de Apoyo',
'Atención al estudiante',
'Bienestar Universitario',
'Calidad académica',
'Educación Virtual',
'Opciones de grado',
'Planta Física',
'Servicios Académicos de Apoyo'
]

In [ ]:
CONECTORES_v1 = [
    "sin embargo", "no obstante", "por otro lado", "ademas", "además",
    "aunque", "pero", "tambien", "también", "por otra parte",
]


## PASO 1: Segmentacion

In [ ]:
# ======================================================================
# PASO 1: Segmentacion
# ======================================================================
print("Cargando spaCy...")
nlp_v1 = spacy.load("es_core_news_lg")


def segmentar_en_clausulas_it1(texto):
    texto = texto.replace("\n", " ")
    doc = nlp_v1(texto)
    clausulas = []
    patron = r'(?=\b(?:' + '|'.join(CONECTORES_v1) + r')\b)'
    for oracion in doc.sents:
        partes = re.split(patron, oracion.text.strip(), flags=re.IGNORECASE)
        for parte in partes:
            parte = parte.strip(" ,.")
            if len(parte) > 5:  # ignoramos fragmentos demasiado cortos/vacios
                clausulas.append(parte)
    return clausulas

## PASO 2: Clasificador de aspecto (zero-shot)

In [ ]:
# ======================================================================
# PASO 2: Clasificador de aspecto (zero-shot, multilingue)
# ======================================================================
print("Cargando modelo zero-shot para aspectos (puede tardar)...")
clasificador_aspecto_v1 = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
)

## PASO 3: Clasificador de sentimiento (BETO)

In [ ]:
# ======================================================================
# PASO 3: Clasificador de sentimiento (BETO, el que ya conoces)
# ======================================================================
print("Cargando BETO para sentimiento...")
clasificador_sentimiento_v1 = pipeline(
    "sentiment-analysis",
    model="finiteautomata/beto-sentiment-analysis",
    top_k=None,  # esto hace que devuelva las 3 probabilidades, no solo la ganadora
)


def elegir_sentimiento_calibrado_it1(scores, margen_minimo=0.15):
    """En vez de quedarnos ciegamente con la etiqueta de mayor puntaje,
    aplicamos una regla: si NEUTRO gana pero por un margen chico frente
    a POS o NEG, preferimos la opcion con carga de sentimiento (evitamos
    que "neutro" se vuelva un cajon de sastre para casos dudosos).
    Ajusta 'margen_minimo' segun lo que veas en tus propios datos."""
    ordenados = sorted(scores, key=lambda x: x["score"], reverse=True)
    ganador = ordenados[0]
    segundo = ordenados[1]

    if ganador["label"] == "NEU" and (ganador["score"] - segundo["score"]) < margen_minimo:
        if segundo["label"] in ("POS", "NEG"):
            return segundo["label"], segundo["score"], True  # True = fue recalibrado
    return ganador["label"], ganador["score"], False


def analizar_comentario_it1(comentario):
    clausulas = segmentar_en_clausulas_it1(comentario)
    resultados = []

    for clausula in clausulas:
        # Aspecto: le pedimos al modelo que elija el mas probable
        resultado_aspecto = clasificador_aspecto_v1(clausula, ASPECTOS_CANDIDATOS_CALIDAD_v1)
        aspecto_principal = resultado_aspecto["labels"][0]
        confianza_aspecto = resultado_aspecto["scores"][0]

        # Sentimiento de ese mismo fragmento: ahora con las 3 probabilidades
        scores_sentimiento = clasificador_sentimiento_v1(clausula)[0]
        etiqueta, confianza, fue_recalibrado = elegir_sentimiento_calibrado_it1(scores_sentimiento)

        resultados.append({
            "fragmento": clausula,
            "aspecto": aspecto_principal,
            "confianza_aspecto": round(confianza_aspecto, 2),
            "sentimiento": etiqueta,
            "confianza_sentimiento": round(confianza, 2),
            "recalibrado": fue_recalibrado,
            "scores_completos": {s["label"]: round(s["score"], 3) for s in scores_sentimiento},
        })

    return resultados

## PRUEBA con 20 comentarios más extensos

In [ ]:
# if __name__ == "__main__":

In [ ]:
for comentario in comentarios_extensos:
    print("\n" + "=" * 70)
    print("COMENTARIO ORIGINAL:")
    print("=" * 70)
    print(comentario)

    resultados = analizar_comentario_it1(comentario)

    print("\n" + "=" * 70)
    print("RESULTADO: sentimiento POR ASPECTO, no uno solo para todo")
    print("=" * 70)
    for r in resultados:
        print(f"\nFragmento: \"{r['fragmento']}\"")
        print(f"  Aspecto:     {r['aspecto']}  (confianza: {r['confianza_aspecto']:.0%})")
        marca = "  <- recalibrado (NEU casi empatado con otra clase)" if r["recalibrado"] else ""
        print(f"  Sentimiento: {r['sentimiento']}  (confianza: {r['confianza_sentimiento']:.0%}){marca}")
        print(f"  Probabilidades completas: {r['scores_completos']}")
    print("=" * 70, "\n\n")

# Iteración 2

> ***Parámetros***:

ASPECTOS_CANDIDATOS:
- etiquetado manual equipo Prisma: `ASPECTOS_CANDIDATOS_CALIDAD_v2`

CONECTORES:
- Listado manual propuesto por Cristina

MODELOS:
- spacy: `es_core_news_lg`
- zeroshot: `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`
- Análisis sentimientos: `pysentimiento/robertuito-sentiment-analysis`

## Configuraciones iteración

In [ ]:
# Etiquetado manual equipo PRISMA
ASPECTOS_CANDIDATOS_CALIDAD_v2 = [
'Calidad Académica Programa',
'Infraestructura Física',
'Modalidad De Grado',
'Monitorias',
'Plagio',
#'Carga Académica',
'Pedagogía',
'Comunicación Con Estudiantes',
'Campus Virtual',
'Bienestar Universitario',
'Materias Optativas',
'Costo Matrícula',
#'Movilidad Social',
'Prácticas Evaluación',
'Exigencia Estudiantes',
'Gestión Quejas',
'Oportunidades Laborales',
'Homologación Materias',
'Inversiones Universidad',
'Atención Psicológica',
'Servicios Administrativos',
'Contenido Materias',
'Conexión Con Egresados',
'Acreditación Programa',
'Política Estímulos Docentes',
'Deserción Estudiantil',
#'Rutas',
'Internacionalización',
'Becas',
'Evaluación Desempeño Docente',
'Doble Titulación',
'Convenios',
'Aprendizaje Extracurricular'
]


In [ ]:
# verificar que los aspectos sean únicos
from collections import Counter

conteo_aspectos = Counter(ASPECTOS_CANDIDATOS_CALIDAD_v2)
aspectos_duplicados = [aspecto for aspecto, cuenta in conteo_aspectos.items() if cuenta > 1]

assert not aspectos_duplicados, f"Hay etiquetas de aspecto duplicadas: {aspectos_duplicados}"
print(f"Verificación OK: {len(ASPECTOS_CANDIDATOS_CALIDAD_v2)} aspectos, todos únicos.")

## PASO 1: Segmentacion

In [ ]:
# Ejecutar una sola vez (o dejar comentado si ya está instalado):
# !python -m spacy download es_core_news_lg

nlp_dependencias = spacy.load("es_core_news_lg")

In [ ]:
DEPS_CLAUSULA_SIEMPRE = {"ROOT", "advcl", "parataxis"}
LONGITUD_MINIMA_TOKENS = 3  # tokens de contenido (sin contar puntuación) por fragmento


def _tiene_sujeto_propio(token):
    return any(
        hijo.dep_ in ("nsubj", "nsubj:pass", "csubj", "csubj:pass")
        for hijo in token.children
    )


def _tiene_copula(token):
    """True si `token` es el predicado de una construcción copulativa
    ('X es/está/parece Y'), es decir, tiene un hijo con dep_ == 'cop'."""
    return any(hijo.dep_ == "cop" for hijo in token.children)

def _es_advcl_complemento(token):
    """True si esta 'advcl' es en realidad un complemento del predicado del
    que cuelga (p. ej. 'alineados CON lo que esperábamos') y no una
    cláusula subordinada genuina.

    Patrón: mark = preposición (ADP, no SCONJ) + verbo en forma FINITA.
    Las subordinadas genuinas con preposición usan infinitivo
    ('PARA evitar...' = cláusula de propósito) y las que llevan una
    conjunción subordinante propiamente dicha (SCONJ: aunque, porque,
    cuando...) tampoco deben excluirse.
    """
    marca = next((h for h in token.children if h.dep_ == "mark"), None)
    if marca is None or marca.pos_ != "ADP":
        return False
    return "Fin" in token.morph.get("VerbForm")


def _es_cabeza_de_clausula(token):
    es_copulativo = _tiene_copula(token)

    # En "X es/está Y" el ROOT sintáctico suele ser el predicado (ADJ, NOUN,
    # PRON, ADV...), no un VERB/AUX. Sin esta excepción esa cláusula nunca
    # calificaba como cabeza y sus tokens se perdían en
    # _cabeza_de_clausula_ancestro (quedaban bajo una raíz que el bucle
    # final de _segmentar_doc nunca visita).
    if not es_copulativo and token.pos_ not in ("VERB", "AUX"):
        return False

    if token.dep_ == "advcl" and _es_advcl_complemento(token):
        return False

    if token.dep_ in DEPS_CLAUSULA_SIEMPRE:
        return True
    if token.dep_ == "conj":
        return es_copulativo or _tiene_sujeto_propio(token)
    return False


def _cabeza_de_clausula_ancestro(token, cabezas):
    actual = token
    while True:
        if actual in cabezas:
            return actual
        if actual.head == actual:  # llegó a la raíz de la oración
            return actual
        actual = actual.head


def _reconstruir(tokens):
    return "".join(t.text_with_ws for t in tokens).strip()


def _segmentar_doc(doc):
    fragmentos = []

    for sent in doc.sents:
        cabezas = {t for t in sent if _es_cabeza_de_clausula(t)}

        if len(cabezas) <= 1:
            # una sola proposición (aunque tenga perífrasis, relativas o
            # complementos con "que") -> no se parte
            texto = sent.text.strip()
            tokens_contenido = [t for t in sent if not t.is_punct]

            if len(tokens_contenido) < LONGITUD_MINIMA_TOKENS and fragmentos:
                # oración degenerada (p. ej. un marcador de lista como "4.")
                # -> se fusiona con el fragmento anterior en vez de quedar
                # suelta y pasar por los clasificadores como si fuera una cláusula
                fragmentos[-1] = f"{fragmentos[-1]} {texto}".strip()
            elif texto:
                fragmentos.append(texto)
            continue

        grupos = {cabeza.i: [] for cabeza in cabezas}
        for token in sent:
            ancestro = _cabeza_de_clausula_ancestro(token, cabezas)
            grupos.setdefault(ancestro.i, []).append(token)

        for cabeza in sorted(cabezas, key=lambda c: c.i):
            tokens_grupo = sorted(grupos.get(cabeza.i, []), key=lambda t: t.i)
            tokens_contenido = [t for t in tokens_grupo if not t.is_punct]
            texto_fragmento = _reconstruir(tokens_grupo)

            if len(tokens_contenido) < LONGITUD_MINIMA_TOKENS and fragmentos:
                # fragmento degenerado -> se fusiona con el anterior
                fragmentos[-1] = f"{fragmentos[-1]} {texto_fragmento}".strip()
            elif texto_fragmento:
                fragmentos.append(texto_fragmento)

    return fragmentos


def segmentar_en_clausulas_it2(comentario):
    doc = nlp_dependencias(comentario)
    return _segmentar_doc(doc)


BATCH_SIZE_DEPENDENCIAS = 100  # separado del BATCH_SIZE de los clasificadores

def segmentar_en_clausulas_it2_batch(comentarios,
                            batch_size=BATCH_SIZE_DEPENDENCIAS, n_process=1):
    return [_segmentar_doc(doc) for doc in nlp_dependencias.pipe(comentarios, batch_size=batch_size, n_process=n_process)]

## PASO 2: Clasificador de aspecto (zero-shot)

In [ ]:
# ======================================================================
# PASO 2: Clasificador de aspecto (zero-shot, multilingue)
# ======================================================================
print("Cargando modelo zero-shot para aspectos (puede tardar)...")
clasificador_aspecto_v1 = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    device=device
)

## PASO 3: Clasificador de sentimiento (BETO)

In [ ]:
# ======================================================================
# PASO 3: Clasificador de sentimiento (BETO, el que ya conoces)
# ======================================================================
print("Cargando Robertuito para sentimiento...")
clasificador_sentimiento_v1 = pipeline(
    "text-classification",
    model="pysentimiento/robertuito-sentiment-analysis",
    top_k=None,  # esto hace que devuelva las 3 probabilidades, no solo la ganadora
    device=device
)


def elegir_sentimiento_calibrado_it2(scores, margen_minimo=0.15):
    """En vez de quedarnos ciegamente con la etiqueta de mayor puntaje,
    aplicamos una regla: si NEUTRO gana pero por un margen chico frente
    a POS o NEG, preferimos la opcion con carga de sentimiento (evitamos
    que "neutro" se vuelva un cajon de sastre para casos dudosos).
    Ajusta 'margen_minimo' segun lo que veas en tus propios datos."""
    ordenados = sorted(scores, key=lambda x: x["score"], reverse=True)
    ganador = ordenados[0]
    segundo = ordenados[1]

    if ganador["label"] == "NEU" and (ganador["score"] - segundo["score"]) < margen_minimo:
        if segundo["label"] in ("POS", "NEG"):
            return segundo["label"], segundo["score"], True  # True = fue recalibrado
    return ganador["label"], ganador["score"], False



In [ ]:
BATCH_SIZE = 32
UMBRAL_MIN_CONFIANZA_ASPECTO = 0.15  # mismo valor que calibres para la función individual
UMBRAL_MIN_MARGEN_ASPECTO = 0.05

def analizar_comentario_it2(comentario):
    clausulas = segmentar_en_clausulas_it2(comentario)

    if not clausulas:
        return []
    # Aspecto: una sola llamada batcheada para todas las cláusulas del comentario
    resultados_aspecto = clasificador_aspecto_v1(
        clausulas,
        ASPECTOS_CANDIDATOS_CALIDAD_v2,
        batch_size=BATCH_SIZE,
        )

    # Sentimiento: también batcheado, con las 3 probabilidades por cláusula
    resultados_sentimiento = clasificador_sentimiento_v1(
        clausulas,
        batch_size=BATCH_SIZE,
        truncation = True
        )
    resultados = []
    for clausula, resultado_aspecto, scores_sentimiento in zip(
        clausulas, resultados_aspecto, resultados_sentimiento
        ):
        aspecto_top = resultado_aspecto["labels"][0]
        confianza_aspecto = resultado_aspecto["scores"][0]

        # margen entre el aspecto ganador y el segundo candidato más probable
        margen_aspecto = (
            confianza_aspecto - resultado_aspecto["scores"][1]
            if len(resultado_aspecto["scores"]) > 1
            else confianza_aspecto
        )

        # reject option: si el modelo no está realmente seguro, no forzamos
        # una etiqueta real de la lista — usamos el catch-all explícito
        if confianza_aspecto < UMBRAL_MIN_CONFIANZA_ASPECTO or margen_aspecto < UMBRAL_MIN_MARGEN_ASPECTO:
            aspecto_final = "OTRO ASPECTO"
        else:
            aspecto_final = aspecto_top

        etiqueta, confianza, fue_recalibrado = elegir_sentimiento_calibrado_it2(scores_sentimiento)

        resultados.append({
            "fragmento": clausula,
            "aspecto": aspecto_final,
            "aspecto_candidato_original": aspecto_top if aspecto_final == "OTRO ASPECTO" else None,
            "confianza_aspecto": round(confianza_aspecto, 2),
            "sentimiento": etiqueta,
            "confianza_sentimiento": round(confianza, 2),
            "recalibrado": fue_recalibrado,
            "scores_completos": {s["label"]: round(s["score"], 3) for s in scores_sentimiento},
        })

    return resultados

In [ ]:
UMBRAL_MIN_CONFIANZA_ASPECTO = 0.15  # mismo valor que calibres para la función individual
UMBRAL_MIN_MARGEN_ASPECTO = 0.05

# función para usar cuando se procesen muchos comentarios a la vez

def analizar_comentarios_it2_batch(comentarios, batch_size=BATCH_SIZE):
    # 1. Segmentar cada comentario y aplanar TODAS las cláusulas de TODOS los
    #    comentarios en una sola lista, recordando a qué comentario pertenece cada una.
    clausulas_por_comentario = segmentar_en_clausulas_it2_batch(comentarios)

    todas_las_clausulas = []
    indice_comentario_por_clausula = []
    for idx_comentario, clausulas in enumerate(clausulas_por_comentario):
        for clausula in clausulas:
            todas_las_clausulas.append(clausula)
            indice_comentario_por_clausula.append(idx_comentario)

    if not todas_las_clausulas:
        return [[] for _ in comentarios]

    # 2. Clasificar aspecto y sentimiento UNA sola vez para todas las cláusulas
    #    de todos los comentarios juntas (aquí está la ganancia real de rendimiento).
    resultados_aspecto = clasificador_aspecto_v1(
        todas_las_clausulas,
        ASPECTOS_CANDIDATOS_CALIDAD_v2,
        batch_size=batch_size,
    )
    resultados_sentimiento = clasificador_sentimiento_v1(
        todas_las_clausulas,
        batch_size=batch_size,
        truncation = True
    )

    # 3. Reagrupar los resultados, uno por comentario, en el mismo orden de entrada
    resultados_por_comentario = [[] for _ in comentarios]

    for clausula, resultado_aspecto, scores_sentimiento, idx_comentario in zip(
        todas_las_clausulas, resultados_aspecto, resultados_sentimiento, indice_comentario_por_clausula
    ):
        aspecto_top = resultado_aspecto["labels"][0]
        confianza_aspecto = resultado_aspecto["scores"][0]

        margen_aspecto = (
            confianza_aspecto - resultado_aspecto["scores"][1]
            if len(resultado_aspecto["scores"]) > 1
            else confianza_aspecto
        )

        if confianza_aspecto < UMBRAL_MIN_CONFIANZA_ASPECTO or margen_aspecto < UMBRAL_MIN_MARGEN_ASPECTO:
            aspecto_final = "OTRO ASPECTO"
        else:
            aspecto_final = aspecto_top

        etiqueta, confianza, fue_recalibrado = elegir_sentimiento_calibrado_it2(scores_sentimiento)

        resultados_por_comentario[idx_comentario].append({
            "fragmento": clausula,
            "aspecto": aspecto_final,
            "aspecto_candidato_original": aspecto_top if aspecto_final == "OTRO ASPECTO" else None,
            "confianza_aspecto": round(confianza_aspecto, 2),
            "sentimiento": etiqueta,
            "confianza_sentimiento": round(confianza, 2),
            "recalibrado": fue_recalibrado,
            "scores_completos": {s["label"]: round(s["score"], 3) for s in scores_sentimiento},
        })

    return resultados_por_comentario

## PRUEBA con 20 comentarios más extensos

In [ ]:
# if __name__ == "__main__":

In [ ]:
# Procesamiento de todos los comentarios en batch
# resultados_totales = analizar_comentarios_it2_batch(df["comentario"].tolist())

In [ ]:
comentario_ejemplo = '1. Muchas de las preguntas en este cuestionario no las conocía.' \
                       'Durante el programa de la especialización, nunca se nos informó ' \
                        'sobre las instancias del estudiante o del profesor, ni sobre programas de intercambio o la posibilidad de ' \
                'participar en programas de educación con otras universidades. Tampoco se mencionaron opciones de investigación, ' \
                'ni conocimos laboratorios o aulas de apoyo para lo que veíamos en clase. ' \
                '2. En la pregunta relacionada con la calidad del programa, califiqué como insatisfecho porque el plan de estudios no estaba ' \
                'actualizado a las tendencias actuales del mundo digital. Además, las materias no estaban relacionadas entre sí, ' \
                'lo que generaba repetición de contenidos. En varias asignaturas (Marketing digital 1, experiencia de usuario, ' \
                'transformación digital, neuromarketing) vimos los mismos temas: funnel, journey map, buyer persona, sin secuencialidad. ' \
                'Claro, son temas muy amplios que seguramente en unas materias hubiéramos visto etapas del funnel o del journey pero no fue así, ' \
                'siempre vimos los mismos temas. Daba la percepción de que la universidad no revisó o entregó la estructura de las clases a los profesores. ' \
                '3. Las materias optativas ofrecidas no estaban alineadas con lo que esperábamos. Tanto así que tuvimos que proponer nuevas asignaturas ' \
                'para cursar. 4. En el primer módulo hubo materias que no se relacionaban directamente con el propósito del programa, ' \
                'como desarrollo sostenible. Aunque los temas fueron interesantes y nos concientizaron sobre la situación del planeta, ' \
                'no encontré una conexión clara con el mundo laboral en áreas digitales, marketing, o los sectores en los que la mayoría ' \
                'de los estudiantes nos desempeñamos. Igualmente, en la materia de liderazgo, es un tema tan relevante para los roles que ' \
                'encontramos hoy en el mercado, pero lo que vimos en esta materia no agregó valor. Sentí que perdimos tiempo con ejercicios ' \
                'como jugar Sudoku o con legos. Tenía expectativas de aprender a enfrentar entornos laborales complejos, generar ambientes ' \
                'productivos y dar buen feedback, pero lo que vimos fueron actividades más apropiadas para compartir en ambientes familiares ' \
                'o amigos. Los tiempos asignados a estas materias se pudieron haber utilizado con temas más afines al ser de la especialización. ' \
                '5. Destaco: 1. Materias muy buenas y profesores excelentes, como por ejemplo: analítica digital, marketing digital II, pensamiento e' \
                'stratégico, entorno y competitividad, neuromarketing. También fue muy interesante los seminarios internacionales (fueron muy cortos).' \
                ' 2. Las instalaciones de la universidad sin dudas son muy lindas, los espacios de estudio, las rutas, la cafetería ' \
                '(podrían ajustarse los horarios de break para evitar largas filas y llegar tarde a las clases). ' \
                'Para cerrar, tenía expectativas más altas del programa, dada la reputación de la universidad. Gracias.'

In [ ]:
print(segmentar_en_clausulas_it2("Tanto así que tuvimos que proponer nuevas asignaturas para cursar."))
# esperado: un solo fragmento, sin partir "tuvimos" de "que proponer"

print(segmentar_en_clausulas_it2(
    "Aunque los temas fueron interesantes y nos concientizaron sobre la situación del planeta, "
    "no encontré una conexión clara con el mundo laboral en áreas digitales, marketing, "
    "o los sectores en los que la mayoría de los estudiantes nos desempeñamos."
))
# esperado: 2 fragmentos — la cláusula concesiva completa, y la cláusula principal completa —
# sin "y nos concientizaron..." aislado ni comas huérfanas

print(segmentar_en_clausulas_it2("Sentí que perdimos tiempo con ejercicios como jugar Sudoku o con legos."))
# esperado: un solo fragmento, sin "Sentí ." suelto

In [ ]:
print(segmentar_en_clausulas_it2(comentario_ejemplo))

In [ ]:
print("\n" + "=" * 70)
print("COMENTARIO ORIGINAL:")
print("=" * 70)
print(comentario_ejemplo)

resultados = analizar_comentario_it2(comentario_ejemplo)

print("\n" + "=" * 70)
print("RESULTADO: sentimiento POR ASPECTO, no uno solo para todo")
print("=" * 70)
for r in resultados:
    print(f"\nFragmento: \"{r['fragmento']}\"")
    print(f"  Aspecto:     {r['aspecto']}  (confianza: {r['confianza_aspecto']:.0%})")
    marca = "  <- recalibrado (NEU casi empatado con otra clase)" if r["recalibrado"] else ""
    print(f"  Sentimiento: {r['sentimiento']}  (confianza: {r['confianza_sentimiento']:.0%}){marca}")
    print(f"  Probabilidades completas: {r['scores_completos']}")
print("=" * 70, "\n\n")

In [ ]:
for comentario in comentarios_extensos:
    print("\n" + "=" * 70)
    print("COMENTARIO ORIGINAL:")
    print("=" * 70)
    print(comentario)

    resultados = analizar_comentario_it2(comentario)

    print("\n" + "=" * 70)
    print("RESULTADO: sentimiento POR ASPECTO, no uno solo para todo")
    print("=" * 70)
    for r in resultados:
        print(f"\nFragmento: \"{r['fragmento']}\"")
        print(f"  Aspecto:     {r['aspecto']}  (confianza: {r['confianza_aspecto']:.0%})")
        marca = "  <- recalibrado (NEU casi empatado con otra clase)" if r["recalibrado"] else ""
        print(f"  Sentimiento: {r['sentimiento']}  (confianza: {r['confianza_sentimiento']:.0%}){marca}")
        print(f"  Probabilidades completas: {r['scores_completos']}")
    print("=" * 70, "\n\n")